In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d


from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import matplotlib

from scipy.interpolate import LinearNDInterpolator
import scipy

from scipy import integrate 

from scipy.stats import gaussian_kde
from sklearn.neighbors import KernelDensity

import eloss_average
import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst


import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle


In [ ]:
print(f'{np.__version__ = }')
print(f'{matplotlib.__version__ = }')
print(f'{pd.__version__ = }')
print(f'{scipy.__version__ = }')

# Read it in

In [ ]:
# First run examine_muons_through_rock.ipynb in the 
# ../geant4_simulations directory
# 
# then make a soft link to muons_summary_from_GEANT4_simulations.parquet

In [ ]:
# We need this file. Make sure there is a soft link to it
infilename_for_eloss = 'muons_summary_from_GEANT4_simulations.parquet'
df_eloss = pd.read_parquet(infilename_for_eloss)
df_eloss

In [ ]:
theta_rad = np.arctan2(-df_eloss['z'], df_eloss['r'])
theta_rad[theta_rad>0] = np.pi/2 - theta_rad
theta_rad[theta_rad<0] = np.pi/2 + theta_rad


theta_deg = np.rad2deg(theta_rad)
costheta = np.cos(theta_rad)

df_eloss['theta_rad'] = theta_rad
df_eloss['theta_deg'] = theta_deg
df_eloss['costheta'] = costheta

df_eloss

In [ ]:
# Make some plots
# Make some plots
e_initials = df_eloss['e_initial'].unique()
print(f'Initial energies: \n{e_initials}')

n_energies = len(e_initials)

ncols = 4
nrows = int(n_energies/ncols)+1

height = nrows*4

plt.figure(figsize=(12,height))

for idx,ei in enumerate(e_initials[0:]):
    #print(f"ei: {ei}")
    filter = df_eloss['e_initial']==ei

    z = df_eloss[filter]['z']
    ef = ei - df_eloss[filter]['e']

    print(f"ei: {ei}   {len(z)}  {len(ef)}")

    plt.subplot(nrows,ncols,idx+1)
    
    #plt.plot(z,ef,'.',markersize=0.1)
    plt.hist2d(z,ef,bins=50, norm='log')
    plt.xlabel('z (m)')
    plt.ylabel('Final energy (GeV)')
    #plt.title(f'Ei: {ei} GeV')

    textstr = f'E$_i$ {ei} GeV'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax = plt.gca()
    ax.text(0.45, 0.95, textstr, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

plt.tight_layout()

In [ ]:
# Make some slices

ei = 1000

n_z = 12
colors = plt.get_cmap('tab20')(np.linspace(0, 1, n_z))
linestyles = ['solid', 'dashed']#, 'dashdot', 'dotted']

#print(colors)


e_initials = df_eloss['e_initial'].unique()
print(f'Initial energies: \n{e_initials}')

n_energies = len(e_initials)

plotsperfig = 5
nfigs = int(n_energies/plotsperfig)+1

ncols = 1
nrows = plotsperfig

height = plotsperfig*4

for idx,ei in enumerate(e_initials[0:]):

    if idx%plotsperfig==0:
        plt.figure(figsize=(12,height))
        print("Made a new figure!")

    filter = df_eloss['e_initial']==ei
    
    z = df_eloss[filter]['z']
    ef = ei - df_eloss[filter]['e']
    
    lo = min(z)
    hi = max(z)
    step = (hi-lo)/n_z

    width = 50
    if width>step:
        width=step
    
    print(f'ei: {ei}    {lo = }   {hi = }')

    ifig = int(idx%5)+1
    plt.subplot(nrows, ncols, ifig )

    for i in range(n_z):

        #print(f"ei: {ei}")
        #filter_z = (z>idx*width) & (z<(idx+1)*width)
        zlo = i*step
        zhi = i*step+width
        filter_z = (z>zlo) & (z<zhi)
    
        eftmp = ef[filter_z]
    
        #print(f"z: {zlo:.0f}  {zhi:.0f}   {len(eftmp)}")
    
        plt.hist(eftmp,bins=100,histtype='step', range=(0,ei), \
                 linewidth=2, \
                 color=colors[i], linestyle=linestyles[i%len(linestyles)], \
                 density=True,label=f'{zlo:.0f}-{zhi:.0f} m')
        plt.xlabel('E (GeV)')
        #plt.ylabel('Final energy (GeV)')
        #plt.title(f'Ei: {ei} GeV')
    
    plt.xlim(0,1.5*ei)
    
    plt.yscale('log')
    plt.legend()
    textstr = f'E$_i$ {ei} GeV'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax = plt.gca()
    ax.text(0.45, 0.95, textstr, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

    if idx%plotsperfig==(plotsperfig-1):
        plt.tight_layout()
        outfile = f'plots/efinal_for_distance_slices_ei_{ei:.0f}.png'
        plt.savefig(outfile)
        print(f'Saved to {outfile}')

# Angular deflections

In [ ]:
# Make some slices

ei = 1000

n_z = 12
colors = plt.get_cmap('tab20')(np.linspace(0, 1, n_z))
linestyles = ['solid', 'dashed']#, 'dashdot', 'dotted']

#print(colors)


e_initials = df_eloss['e_initial'].unique()
print(f'Initial energies: \n{e_initials}')

n_energies = len(e_initials)

plotsperfig = 5
nfigs = int(n_energies/plotsperfig)+1

ncols = 1
nrows = plotsperfig

height = plotsperfig*4

for idx,ei in enumerate(e_initials[0:]):

    if idx%plotsperfig==0:
        plt.figure(figsize=(12,height))
        print("Made a new figure!")

    filter = df_eloss['e_initial']==ei
    
    z = df_eloss[filter]['z']
    theta_deg = df_eloss[filter]['theta_deg']
    
    lo = min(z)
    hi = max(z)
    step = (hi-lo)/n_z

    width = 50
    if width>step:
        width=step
    
    print(f'ei: {ei}    {lo = }   {hi = }')

    ifig = int(idx%5)+1
    plt.subplot(nrows, ncols, ifig )

    for i in range(n_z):

        #print(f"ei: {ei}")
        #filter_z = (z>idx*width) & (z<(idx+1)*width)
        zlo = i*step
        zhi = i*step+width
        filter_z = (z>zlo) & (z<zhi)
    
        tdtmp = theta_deg[filter_z]
    
        #print(f"z: {zlo:.0f}  {zhi:.0f}   {len(eftmp)}")
    
        plt.hist(tdtmp, bins=100, histtype='step', range=(0,1), \
                 linewidth=2, \
                 color=colors[i], linestyle=linestyles[i%len(linestyles)], \
                 density=True,label=f'{zlo:.0f}-{zhi:.0f} m')
        plt.xlabel(r'$\theta$ (degrees)')
        #plt.ylabel('Final energy (GeV)')
        #plt.title(f'Ei: {ei} GeV')
    
    plt.xlim(0,1.5)
    
    plt.yscale('log')
    plt.legend()
    textstr = f'E$_i$ {ei} GeV'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax = plt.gca()
    ax.text(0.45, 0.95, textstr, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

    if idx%plotsperfig==(plotsperfig-1):
        plt.tight_layout()
        outfile = f'plots/theta_for_distance_slices_ei_{ei:.0f}.png'
        plt.savefig(outfile)
        print(f'Saved to {outfile}')

In [ ]:
# Make some slices

ei = 1000

n_z = 12
colors = plt.get_cmap('tab20')(np.linspace(0, 1, n_z))
linestyles = ['solid', 'dashed']#, 'dashdot', 'dotted']

#print(colors)

e_initials = df_eloss['e_initial'].unique()
print(f'Initial energies: \n{e_initials}')

n_energies = len(e_initials)

plotsperfig = 5
nfigs = int(n_energies/plotsperfig)+1

ncols = 1
nrows = plotsperfig

height = plotsperfig*4

for idx,ei in enumerate(e_initials[0:]):

    if idx%plotsperfig==0:
        plt.figure(figsize=(12,height))
        print("Made a new figure!")

    filter = df_eloss['e_initial']==ei
    
    z = df_eloss[filter]['z']
    r = df_eloss[filter]['r']
    
    lo = min(z)
    hi = max(z)
    step = (hi-lo)/n_z

    width = 50
    if width>step:
        width=step
    
    print(f'ei: {ei}    {lo = }   {hi = }')

    ifig = int(idx%5)+1
    plt.subplot(nrows, ncols, ifig )

    for i in range(n_z):

        #print(f"ei: {ei}")
        #filter_z = (z>idx*width) & (z<(idx+1)*width)
        zlo = i*step
        zhi = i*step+width
        filter_z = (z>zlo) & (z<zhi)
    
        rtmp = r[filter_z]
    
        #print(f"z: {zlo:.0f}  {zhi:.0f}   {len(eftmp)}")
    
        plt.hist(rtmp, bins=100, histtype='step', range=(0,10), \
                 linewidth=2, \
                 color=colors[i], linestyle=linestyles[i%len(linestyles)], \
                 density=True,label=f'{zlo:.0f}-{zhi:.0f} m')
        plt.xlabel(r'Transverse displacement (m)')
        #plt.ylabel('Final energy (GeV)')
        #plt.title(f'Ei: {ei} GeV')
    
    plt.xlim(0,15)
    
    plt.yscale('log')
    plt.legend()
    textstr = f'E$_i$ {ei} GeV'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax = plt.gca()
    ax.text(0.45, 0.95, textstr, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

    if idx%plotsperfig==(plotsperfig-1):
        plt.tight_layout()
        outfile = f'plots/dt_for_distance_slices_ei_{ei:.0f}.png'
        plt.savefig(outfile)
        print(f'Saved to {outfile}')

# Build up the CDFs

In [ ]:
from scipy import integrate
from scipy.interpolate import interp1d, make_smoothing_spline
from scipy.interpolate import CubicSpline


In [ ]:
def interpolate_2d_quantile(x11, y11,   # E1, d1
                             x12, y12,   # E1, d2
                             x21, y21,   # E2, d1
                             x22, y22,   # E2, d2
                             fe, fd,     # fractional position in E and d
                             npts=500):
    """
    Bilinear quantile interpolation over a 2D (energy, distance) grid.
    fe=0 -> E1, fe=1 -> E2; fd=0 -> d1, fd=1 -> d2
    All input y's should be normalizedac PDFs.
    """
    q_grid = np.linspace(0.001, 0.999, npts)

    def make_quantile_fn(x, y):
        cdf = integrate.cumulative_trapezoid(y, x, initial=0)
        cdf /= cdf[-1]
        return interp1d(cdf, x, bounds_error=False,
                        fill_value=(x[0], x[-1]))

    Q11 = make_quantile_fn(x11, y11)
    Q12 = make_quantile_fn(x12, y12)
    Q21 = make_quantile_fn(x21, y21)
    Q22 = make_quantile_fn(x22, y22)

    # Bilinear combination of quantile functions
    w11 = (1 - fe) * (1 - fd)
    w12 = (1 - fe) * fd
    w21 = fe       * (1 - fd)
    w22 = fe       * fd

    x_at_q = (w11 * Q11(q_grid) + w12 * Q12(q_grid) +
               w21 * Q21(q_grid) + w22 * Q22(q_grid))

    # Differentiate to get PDF
    x_mid = 0.5 * (x_at_q[:-1] + x_at_q[1:])
    dq    = np.diff(q_grid)
    dx    = np.diff(x_at_q)
    y_out = dq / np.abs(dx)

    norm = integrate.trapezoid(y_out, x_mid)
    return x_mid, y_out / norm

In [ ]:
E1 = 10000
E2 = 20000

slice_width = 10 # Actually width is 2x this value
d1 = 500
d2 = 900

efilter1 = df_eloss['e_initial']==E1
efilter2 = df_eloss['e_initial']==E2

dfilter1 = (df_eloss['z']>(d1-slice_width)) & (df_eloss['z']<(d1+slice_width))
dfilter2 = (df_eloss['z']>(d2-slice_width)) & (df_eloss['z']<(d2+slice_width))

xpts11 = df_eloss[efilter1 & dfilter1]['e'].values
xpts12 = df_eloss[efilter1 & dfilter2]['e'].values
xpts21 = df_eloss[efilter2 & dfilter1]['e'].values
xpts22 = df_eloss[efilter2 & dfilter2]['e'].values

counts11,bin_edges11 = np.histogram(xpts11, bins=20, density=True)
counts12,bin_edges12 = np.histogram(xpts12, bins=20, density=True)
counts21,bin_edges21 = np.histogram(xpts21, bins=20, density=True)
counts22,bin_edges22 = np.histogram(xpts22, bins=20, density=True)

plt.hist(xpts11,bins=100, range=(0,E2), alpha=0.5, label='11')
plt.hist(xpts12,bins=100, range=(0,E2), alpha=0.5, label='12')
plt.hist(xpts21,bins=100, range=(0,E2), alpha=0.5, label='21')
plt.hist(xpts22,bins=100, range=(0,E2), alpha=0.5, label='22')

plt.legend()

def get_bin_centers(bin_edges):
    width = bin_edges[1] - bin_edges[0]
    return bin_edges[0:-1] + width/2

bc11 = get_bin_centers(bin_edges11)
bc12 = get_bin_centers(bin_edges12)
bc21 = get_bin_centers(bin_edges21)
bc22 = get_bin_centers(bin_edges22)

nspline_pts = 100
lam = 0.02

#int11 = interp1d(bc11, counts11, kind='quadratic')
#int11 = interp1d(bc11, counts11, kind='cubic')
#int11 = CubicSpline(bc11, counts11)
int11 = make_smoothing_spline(bc11, counts11, lam=lam)
x11 = np.linspace(min(bc11), max(bc11), nspline_pts)
y11 = int11(x11)

#int12 = CubicSpline(bc12, counts12)
int12 = make_smoothing_spline(bc12, counts12, lam=lam)
x12 = np.linspace(min(bc12), max(bc12), nspline_pts)
y12 = int12(x12)

#int21 = CubicSpline(bc21, counts21)
int21 = make_smoothing_spline(bc21, counts21, lam=lam)
x21 = np.linspace(min(bc21), max(bc21), nspline_pts)
y21 = int21(x21)

#int22 = CubicSpline(bc22, counts22)
int22 = make_smoothing_spline(bc22, counts22, lam=lam)
x22 = np.linspace(min(bc22), max(bc22), nspline_pts)
y22 = int22(x22)


plt.figure(figsize=(8,8))
plt.subplot(2,2,1)
plt.plot(bc11, counts11)
plt.plot(x11, y11)

plt.subplot(2,2,2)
plt.plot(bc12, counts12)
plt.plot(x12, y12)

plt.subplot(2,2,3)
plt.plot(bc21, counts21)
plt.plot(x21, y21)

plt.subplot(2,2,4)
plt.plot(bc22, counts22)
plt.plot(x22, y22)



fe = 0.9
fd = 0.01
xmid, ymid = interpolate_2d_quantile(x11, y11,   # E1, d1
                             x12, y12,   # E1, d2
                             x21, y21,   # E2, d1
                             x22, y22,   # E2, d2
                             fe, fd,     # fractional position in E and d
                             npts=500)


plt.figure()
plt.plot(x11, y11,label='11')
plt.plot(x12, y12,label='12')
plt.plot(x21, y21,label='21')
plt.plot(x22, y22,label='22')

plt.plot(xmid,ymid,lw=4,label='interp')
plt.legend()

;

In [ ]:
bin_edges22

In [ ]:
xpts11

In [ ]:
df_eloss[efilter1]